# M1: Exploratory Data Analysis

This notebook examines the synthetic monthly marketing dataset.

The cleaned dataset contains 36 months from January 2023 to
December 2025. Exploratory analysis uses the first 30 months.
The final 6 months are reserved for model holdout evaluation.

This section covers:
- Data overview
- Summary statistics
- Potential outliers using the 1.5 × IQR rule

Outlier flags identify observations for review. They do not
automatically justify deleting or changing data.

The data is synthetic; findings are illustrative.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Find the repository root whether the notebook starts
# from the root directory or the notebooks directory.
current = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path
        for path in [current, *current.parents]
        if (path / "src" / "config.py").is_file()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Cannot find the repository root containing src/config.py"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config

TABLE_DIR = config.OUT / "tables" / "m1"
FIGURE_DIR = config.FIG_DIR / "m1"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40)

print("Project root:", PROJECT_ROOT)
print("Tables:", TABLE_DIR)
print("Figures:", FIGURE_DIR)

In [ ]:
df = pd.read_csv(
    config.CLEAN_CSV,
    parse_dates=["month"],
)

df = df.sort_values("month").reset_index(drop=True)

# Verify that this is the expected complete monthly dataset.
assert len(df) == 36
assert df["month"].is_unique
assert not df.isna().any().any()

expected_dates = pd.date_range(
    start="2023-01-01",
    periods=36,
    freq="MS",
)
assert pd.DatetimeIndex(df["month"]).equals(expected_dates)

# Reserve the final 6 months for model evaluation.
eda = df.iloc[:-6].copy()

print("Full cleaned dataset:", df.shape)
print("EDA dataset:", eda.shape)
print(
    "EDA period:",
    eda["month"].min().date(),
    "to",
    eda["month"].max().date(),
)
print("Reserved holdout period: 2025-07-01 to 2025-12-01")

display(eda.head())

In [ ]:
analysis_columns = (
    [config.TARGET, config.TOTAL_SPEND]
    + config.CHANNELS
    + config.CONTROLS
)

summary = eda[analysis_columns].describe().T
summary.index.name = "variable"

summary_path = TABLE_DIR / "summary_statistics_train.csv"
summary.to_csv(summary_path)

display(summary.round(3))
print("Saved:", summary_path.relative_to(PROJECT_ROOT))

In [ ]:
outlier_records = []
threshold_records = []

for column in analysis_columns:
    values = eda[column]

    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    mask = (values < lower_bound) | (values > upper_bound)

    threshold_records.append(
        {
            "variable": column,
            "q1": q1,
            "q3": q3,
            "iqr": iqr,
            "lower_bound": lower_bound,
            "upper_bound": upper_bound,
            "flagged_count": int(mask.sum()),
        }
    )

    for _, row in eda.loc[mask, ["month", column]].iterrows():
        outlier_records.append(
            {
                "month": row["month"],
                "variable": column,
                "value": row[column],
                "lower_bound": lower_bound,
                "upper_bound": upper_bound,
                "review_status": "pending_review",
                "action": "retained",
            }
        )

thresholds = pd.DataFrame(threshold_records)

outliers = pd.DataFrame(
    outlier_records,
    columns=[
        "month",
        "variable",
        "value",
        "lower_bound",
        "upper_bound",
        "review_status",
        "action",
    ],
)

thresholds.to_csv(
    TABLE_DIR / "outlier_thresholds_train.csv",
    index=False,
)

outliers.to_csv(
    TABLE_DIR / "outlier_flags_train.csv",
    index=False,
    date_format="%Y-%m-%d",
)

display(
    thresholds[["variable", "flagged_count"]]
    .sort_values("flagged_count", ascending=False)
    .reset_index(drop=True)
)

if outliers.empty:
    print("No values flagged by the 1.5 × IQR rule.")
else:
    display(outliers)
    print("Flagged values:", len(outliers))
    print("Distinct months flagged:", outliers["month"].nunique())

print("No observations were removed or changed.")

## Outlier screening findings

The 1.5 × IQR screening of the first 30 months flagged one value:
Google Search spend in September 2023 was LKR 5,096,864, above
the upper screening threshold of LKR 4,567,065.13.

No values were flagged in the other 15 variables examined.

The flagged observation is retained without capping or removal.
An IQR flag alone does not establish a data error, and no source
evidence identifies this value as erroneous or explains its cause.

The screening does not adjust for seasonality or time trends.
Absence of an IQR flag does not prove absence of anomalies.

In [ ]:
reviewed_flags = pd.read_csv(
    TABLE_DIR / "outlier_flags_train.csv",
    parse_dates=["month"],
)

reviewed_flags["review_note"] = ""

review_mask = (
    reviewed_flags["month"].eq(pd.Timestamp("2023-09-01"))
    & reviewed_flags["variable"].eq("google_search_spend_lkr")
)

reviewed_flags.loc[
    review_mask, "review_status"
] = "reviewed_retained"

reviewed_flags.loc[review_mask, "review_note"] = (
    "IQR flag alone does not establish an error. "
    "No source evidence supports correction or removal; cause unknown."
)

reviewed_flags.to_csv(
    TABLE_DIR / "outlier_flags_train.csv",
    index=False,
    date_format="%Y-%m-%d",
)

display(reviewed_flags)

In [ ]:
import matplotlib.dates as mdates

fig, axes = plt.subplots(
    2, 1,
    figsize=(12, 7),
    sharex=True,
    constrained_layout=True,
)

axes[0].plot(
    eda["month"],
    eda[config.TARGET] / 1_000_000,
    marker="o",
    color="#2563EB",
)
axes[0].set_title("Monthly sales revenue")
axes[0].set_ylabel("Revenue (LKR million)")

axes[1].plot(
    eda["month"],
    eda[config.TOTAL_SPEND] / 1_000_000,
    marker="o",
    color="#059669",
)
axes[1].set_title("Monthly total advertising spend")
axes[1].set_ylabel("Spend (LKR million)")
axes[1].set_xlabel("Month")

axes[1].xaxis.set_major_locator(
    mdates.MonthLocator(interval=3)
)
axes[1].xaxis.set_major_formatter(
    mdates.DateFormatter("%b %Y")
)
plt.setp(
    axes[1].get_xticklabels(),
    rotation=45,
    ha="right",
)

fig.suptitle(
    "Revenue and advertising spend | Jan 2023–Jun 2025\n"
    "Synthetic data; final six months excluded",
    fontsize=13,
)

fig.savefig(
    FIGURE_DIR / "01_revenue_spend_trends_train.png",
    dpi=200,
    bbox_inches="tight",
)

plt.show()
plt.close(fig)

In [ ]:
import calendar

seasonality = (
    eda.groupby("month_num")[config.TARGET]
    .agg(
        observations="count",
        mean_revenue_lkr="mean",
        median_revenue_lkr="median",
    )
    .reindex(range(1, 13))
    .reset_index()
)

seasonality.to_csv(
    TABLE_DIR / "seasonality_summary_train.csv",
    index=False,
)

fig, ax = plt.subplots(
    figsize=(12, 5),
    constrained_layout=True,
)

ax.bar(
    seasonality["month_num"],
    seasonality["mean_revenue_lkr"] / 1_000_000,
    color="#93C5FD",
    label="Mean revenue",
)

ax.scatter(
    eda["month_num"],
    eda[config.TARGET] / 1_000_000,
    color="#1E3A8A",
    s=35,
    zorder=3,
    label="Individual monthly observations",
)

ax.set_xticks(range(1, 13))
ax.set_xticklabels(
    [
        f"{calendar.month_abbr[int(row.month_num)]}\n"
        f"n={int(row.observations)}"
        for row in seasonality.itertuples()
    ]
)

ax.set_ylabel("Revenue (LKR million)")
ax.set_xlabel("Calendar month and observation count")
ax.set_title(
    "Revenue by calendar month | Jan 2023–Jun 2025\n"
    "Descriptive pattern; not adjusted for trend or promotions"
)
ax.legend()

fig.savefig(
    FIGURE_DIR / "02_monthly_revenue_pattern_train.png",
    dpi=200,
    bbox_inches="tight",
)

plt.show()
plt.close(fig)

display(seasonality.round(2))

## Revenue and calendar-month patterns

Monthly revenue fluctuates rather than showing a consistently
increasing trend. Advertising spend is generally lower during
much of 2024 than in 2023 and rises again in early 2025.

Revenue and advertising spend do not move together in every month.
These plots alone cannot separate advertising effects from
seasonality, promotions, other controls or carry-over effects.

April and December have the highest calendar-month mean revenues
in the exploratory sample, both approximately LKR 96 million.

These are descriptive patterns, not isolated seasonal effects.
January–June each have three observations, while July–December
each have two. The small, unequal samples and lack of adjustment
for trend and promotions limit interpretation.

In [ ]:
CHANNEL_LABELS = {
    "tv_spend_lkr": "TV",
    "facebook_instagram_spend_lkr": "Facebook / Instagram",
    "google_search_spend_lkr": "Google Search",
    "youtube_spend_lkr": "YouTube",
    "radio_spend_lkr": "Radio",
    "outdoor_spend_lkr": "Outdoor",
}

channel_colors = dict(
    zip(
        config.CHANNELS,
        sns.color_palette("colorblind", len(config.CHANNELS)),
    )
)

fig, axes = plt.subplots(
    3, 2,
    figsize=(13, 10),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)

for ax, channel in zip(axes.flat, config.CHANNELS):
    ax.plot(
        eda["month"],
        eda[channel] / 1_000_000,
        marker="o",
        markersize=4,
        color=channel_colors[channel],
    )

    ax.set_title(CHANNEL_LABELS[channel])
    ax.set_ylabel("Spend (LKR million)")
    ax.set_ylim(bottom=0)

    ax.xaxis.set_major_locator(
        mdates.MonthLocator(interval=6)
    )
    ax.xaxis.set_major_formatter(
        mdates.DateFormatter("%b %Y")
    )

    # Highlight the previously reviewed IQR flag.
    if channel == "google_search_spend_lkr":
        flagged = eda[
            eda["month"].eq(pd.Timestamp("2023-09-01"))
        ]
        ax.scatter(
            flagged["month"],
            flagged[channel] / 1_000_000,
            s=100,
            facecolors="none",
            edgecolors="red",
            linewidths=2,
            label="Reviewed IQR flag; retained",
            zorder=5,
        )
        ax.legend(fontsize=8)

for ax in axes[-1]:
    ax.set_xlabel("Month")
    plt.setp(
        ax.get_xticklabels(),
        rotation=45,
        ha="right",
    )

fig.suptitle(
    "Advertising spend by channel | Jan 2023–Jun 2025\n"
    "Synthetic data; shared vertical scale",
    fontsize=14,
)

fig.savefig(
    FIGURE_DIR / "03_channel_spend_trends_train.png",
    dpi=200,
    bbox_inches="tight",
)

plt.show()
plt.close(fig)

In [ ]:
channel_spend = eda[config.CHANNELS]
monthly_total = channel_spend.sum(axis=1)

spend_mix_pct = (
    channel_spend.div(monthly_total, axis=0) * 100
)

np.testing.assert_allclose(
    spend_mix_pct.sum(axis=1),
    100.0,
)

mix_output = spend_mix_pct.copy()
mix_output.columns = [
    f"{channel}_share_pct"
    for channel in config.CHANNELS
]
mix_output.insert(0, "month", eda["month"].to_numpy())

mix_output.to_csv(
    TABLE_DIR / "monthly_spend_mix_train.csv",
    index=False,
    date_format="%Y-%m-%d",
)

fig, ax = plt.subplots(
    figsize=(13, 6),
    constrained_layout=True,
)

ax.stackplot(
    eda["month"],
    *[
        spend_mix_pct[channel].to_numpy()
        for channel in config.CHANNELS
    ],
    labels=[
        CHANNEL_LABELS[channel]
        for channel in config.CHANNELS
    ],
    colors=[
        channel_colors[channel]
        for channel in config.CHANNELS
    ],
    alpha=0.9,
)

ax.set_ylim(0, 100)
ax.set_xlim(eda["month"].min(), eda["month"].max())
ax.set_ylabel("Share of monthly channel spend (%)")
ax.set_xlabel("Month")
ax.set_title(
    "Advertising spend mix | Jan 2023–Jun 2025\n"
    "Synthetic data; shares use the sum of six channel spends"
)

ax.xaxis.set_major_locator(
    mdates.MonthLocator(interval=3)
)
ax.xaxis.set_major_formatter(
    mdates.DateFormatter("%b %Y")
)
plt.setp(
    ax.get_xticklabels(),
    rotation=45,
    ha="right",
)

ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.22),
    ncol=3,
    frameon=False,
)

fig.savefig(
    FIGURE_DIR / "04_channel_spend_mix_train.png",
    dpi=200,
    bbox_inches="tight",
)

plt.show()
plt.close(fig)

## Channel spend and allocation patterns

TV and Facebook/Instagram generally receive larger monthly
allocations than the other individual channels. Radio has
comparatively low spending throughout the exploratory period.

Several channels show lower spending during parts of 2024,
followed by increases in early 2025. These shared movements
motivate checking correlations between channel spends.

Google Search spend in September 2023 is highlighted as the
previously reviewed IQR flag and remains unchanged.

The allocation mix varies over time, although TV and
Facebook/Instagram together frequently account for approximately
half of monthly channel expenditure.

Spend levels and budget shares describe allocation, not channel
effectiveness or ROI.

In [ ]:
# Revenue, six channel spends and eight controls.
# Exclude totals, derived features and leakage columns.
correlation_columns = (
    [config.TARGET]
    + config.CHANNELS
    + config.CONTROLS
)

LABELS = {
    **CHANNEL_LABELS,
    config.TARGET: "Revenue",
    "average_price_index": "Price index",
    "promotion_discount_pct": "Promotion discount",
    "competitor_pressure_index_0_100": "Competitor pressure",
    "economic_sentiment_index_0_100": "Economic sentiment",
    "lkr_usd_exchange_rate": "Exchange rate",
    "inflation_proxy_index": "Inflation proxy",
    "seasonal_demand_index": "Seasonal demand",
    "public_holiday_count": "Public holidays",
}

correlation = eda[correlation_columns].corr(method="pearson")

correlation.to_csv(
    TABLE_DIR / "pearson_correlations_train.csv",
    index_label="variable",
)

plot_correlation = correlation.rename(
    index=LABELS,
    columns=LABELS,
)

# Hide the repeated upper triangle.
mask = np.triu(
    np.ones_like(plot_correlation, dtype=bool),
    k=1,
)

fig, ax = plt.subplots(figsize=(14, 11))

sns.heatmap(
    plot_correlation,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    linewidths=0.4,
    annot_kws={"size": 8},
    cbar_kws={"label": "Pearson correlation (r)", "shrink": 0.8},
    ax=ax,
)

ax.set_title(
    "Correlations | Jan 2023–Jun 2025 (n=30)\n"
    "Raw same-month associations; synthetic data",
    pad=18,
)

plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
plt.setp(ax.get_yticklabels(), rotation=0)

fig.tight_layout()

fig.savefig(
    FIGURE_DIR / "05_correlation_heatmap_train.png",
    dpi=200,
    bbox_inches="tight",
)

plt.show()
plt.close(fig)

In [ ]:
fig, axes = plt.subplots(
    3, 2,
    figsize=(13, 11),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)

correlation_records = []

for ax, channel in zip(axes.flat, config.CHANNELS):
    x = eda[channel] / 1_000_000
    y = eda[config.TARGET] / 1_000_000

    r = eda[channel].corr(eda[config.TARGET])

    ax.scatter(
        x,
        y,
        color=channel_colors[channel],
        alpha=0.8,
        s=45,
        edgecolors="white",
        linewidths=0.5,
    )

    ax.set_title(
        f"{CHANNEL_LABELS[channel]} | Pearson r = {r:.2f}"
    )
    ax.set_xlabel("Channel spend (LKR million)")
    ax.set_ylabel("Revenue (LKR million)")
    ax.set_xlim(left=0)

    if channel == "google_search_spend_lkr":
        flagged = eda[
            eda["month"].eq(pd.Timestamp("2023-09-01"))
        ]

        ax.scatter(
            flagged[channel] / 1_000_000,
            flagged[config.TARGET] / 1_000_000,
            s=120,
            facecolors="none",
            edgecolors="red",
            linewidths=2,
            label="Reviewed IQR flag; retained",
        )
        ax.legend(fontsize=8)

    correlation_records.append(
        {
            "channel": channel,
            "pearson_r": r,
            "observations": len(eda),
        }
    )

channel_revenue_correlations = pd.DataFrame(
    correlation_records
)

channel_revenue_correlations.to_csv(
    TABLE_DIR / "channel_revenue_correlations_train.csv",
    index=False,
)

fig.suptitle(
    "Channel spend versus revenue | Jan 2023–Jun 2025\n"
    "Same-month observations; no adjustment for controls or carry-over",
    fontsize=14,
)

fig.savefig(
    FIGURE_DIR / "06_spend_revenue_scatter_train.png",
    dpi=200,
    bbox_inches="tight",
)

plt.show()
plt.close(fig)

display(channel_revenue_correlations.round(3))

## Channel–revenue correlations

Across the 30 exploratory observations, Outdoor has the largest
raw same-month correlation with revenue (r = 0.432).

Google Search and TV have almost identical correlations
(r = 0.360 and 0.359). YouTube and Facebook/Instagram also have
similar correlations (r = 0.308 and 0.307).

Radio has a correlation close to zero (r = -0.055), indicating
little same-month linear association in this sample.

These correlations do not adjust for seasonality, promotions,
other controls, carry-over effects or relationships between
channels. They are not causal effects or ROI estimates and
must not be used alone to rank channel effectiveness.

In [ ]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import (
    variance_inflation_factor,
)

# Use only the six raw channel spend variables.
X = eda[config.CHANNELS].astype(float).copy()

if not np.isfinite(X.to_numpy()).all():
    raise ValueError("Channel data contains missing or infinite values")

constant_channels = X.columns[X.nunique() <= 1].tolist()
if constant_channels:
    raise ValueError(
        f"VIF is undefined for constant channels: {constant_channels}"
    )

# Standardisation improves numerical conditioning.
# It does not change VIF when an intercept is included.
X_standardised = (X - X.mean()) / X.std(ddof=0)

# Include an intercept in the auxiliary regressions.
X_design = sm.add_constant(
    X_standardised,
    has_constant="add",
)

vif_records = []

for channel in config.CHANNELS:
    column_position = X_design.columns.get_loc(channel)

    vif_value = variance_inflation_factor(
        X_design.to_numpy(),
        column_position,
    )

    vif_records.append(
        {
            "channel": channel,
            "vif": float(vif_value),
            "observations": len(X),
            "predictor_set": "six raw channel spends with intercept",
        }
    )

vif_table = (
    pd.DataFrame(vif_records)
    .sort_values("vif", ascending=False)
    .reset_index(drop=True)
)

vif_table.to_csv(
    TABLE_DIR / "channel_vif_train.csv",
    index=False,
)

display(vif_table[["channel", "vif", "observations"]].round(3))

# Plot finite values. Infinite VIFs remain recorded in the CSV.
if np.isfinite(vif_table["vif"]).all():
    plot_data = vif_table.sort_values("vif")

    fig, ax = plt.subplots(
        figsize=(10, 5),
        constrained_layout=True,
    )

    ax.barh(
        [
            CHANNEL_LABELS[channel]
            for channel in plot_data["channel"]
        ],
        plot_data["vif"],
        color=[
            channel_colors[channel]
            for channel in plot_data["channel"]
        ],
    )

    ax.set_xlabel("Variance Inflation Factor (VIF)")
    ax.set_title(
        "Channel multicollinearity | Jan 2023–Jun 2025\n"
        "Six raw channel spends; intercept included; n=30"
    )
    ax.set_xlim(left=0)

    fig.savefig(
        FIGURE_DIR / "07_channel_vif_train.png",
        dpi=200,
        bbox_inches="tight",
    )

    plt.show()
    plt.close(fig)
else:
    print(
        "Non-finite VIF detected. Inspect linear dependencies "
        "between channels; values are preserved in the CSV."
    )

## Channel multicollinearity

VIF values for the six raw channel-spend variables range from
1.534 to 2.255 across the 30 exploratory observations.

TV has the highest VIF (2.255), followed by YouTube (2.073)
and Outdoor (2.002). These results do not suggest severe linear
multicollinearity within this channel-only predictor set.

No channels are removed based on this diagnostic.

An intercept was included in the auxiliary regressions.
Controls, total spend and derived spend shares were excluded.
M3 and M4 should reassess multicollinearity using their actual
model inputs, including controls and transformed channel spends.

In [ ]:
import warnings

from statsmodels.tsa.stattools import adfuller, kpss

stationarity_columns = [config.TARGET] + config.CHANNELS

ALPHA = 0.05
ADF_MAX_LAG = 3

stationarity_records = []

for column in stationarity_columns:
    series = eda[column].astype(float)

    if not np.isfinite(series.to_numpy()).all():
        raise ValueError(f"{column}: missing or infinite values")

    if series.nunique() <= 1:
        raise ValueError(f"{column}: constant series")

    # ADF: intercept, no deterministic trend.
    # AIC selects a lag from 0 to 3.
    adf_result = adfuller(
        series,
        maxlag=ADF_MAX_LAG,
        regression="c",
        autolag="AIC",
    )

    adf_stat = adf_result[0]
    adf_p = adf_result[1]
    adf_lags = adf_result[2]
    adf_nobs = adf_result[3]

    # Record KPSS warnings instead of losing them.
    with warnings.catch_warnings(record=True) as captured:
        warnings.simplefilter("always")

        kpss_stat, kpss_p, kpss_lags, _ = kpss(
            series,
            regression="c",
            nlags="auto",
        )

    warning_text = " | ".join(
        str(item.message) for item in captured
    )

    # KPSS may return a boundary of its tabulated p-value range.
    if any(
        "actual p-value is greater" in str(item.message)
        for item in captured
    ):
        kpss_p_display = f"> {kpss_p:.2f}"
    elif any(
        "actual p-value is smaller" in str(item.message)
        for item in captured
    ):
        kpss_p_display = f"< {kpss_p:.2f}"
    else:
        kpss_p_display = f"{kpss_p:.4f}"

    adf_reject = adf_p < ALPHA
    kpss_reject = kpss_p < ALPHA

    if adf_reject and not kpss_reject:
        interpretation = "Evidence consistent with level stationarity"
    elif not adf_reject and kpss_reject:
        interpretation = "Evidence against level stationarity"
    elif adf_reject and kpss_reject:
        interpretation = "Conflicting results; investigate further"
    else:
        interpretation = "Inconclusive; neither null rejected"

    stationarity_records.append(
        {
            "variable": column,
            "input_observations": len(series),
            "regression": "c",
            "adf_maxlag": ADF_MAX_LAG,
            "adf_autolag": "AIC",
            "adf_statistic": adf_stat,
            "adf_p_value": adf_p,
            "adf_selected_lags": adf_lags,
            "adf_effective_observations": adf_nobs,
            "adf_reject_unit_root_5pct": adf_reject,
            "kpss_statistic": kpss_stat,
            "kpss_returned_p_value": kpss_p,
            "kpss_p_display": kpss_p_display,
            "kpss_selected_lags": kpss_lags,
            "kpss_reject_level_stationarity_5pct": kpss_reject,
            "interpretation": interpretation,
            "warnings": warning_text,
        }
    )

stationarity_table = pd.DataFrame(stationarity_records)

stationarity_path = (
    TABLE_DIR / "stationarity_tests_train.csv"
)
stationarity_table.to_csv(
    stationarity_path,
    index=False,
)

display(
    stationarity_table[
        [
            "variable",
            "adf_p_value",
            "adf_selected_lags",
            "kpss_p_display",
            "interpretation",
        ]
    ].round({"adf_p_value": 4})
)

print("Saved:", stationarity_path.relative_to(PROJECT_ROOT))
print("Exploratory results only; no data transformations applied.")

## Stationarity test specification and limitations

ADF and KPSS tests are applied to revenue and the six raw channel
spends using the first 30 months.

Both tests use an intercept without a deterministic trend.
ADF selects a lag from 0–3 using AIC. KPSS uses automatic
bandwidth selection. Decisions use a nominal 5% level.

These are exploratory, unadjusted multiple tests with a small
sample. Results are sensitive to lag choices, trends, seasonality
and structural changes. Neither test directly establishes the
absence of seasonal patterns or validates the final MMM.

Failure to reject a null hypothesis is not proof that it is true.
KPSS p-values outside its tabulated range are reported as bounds.

No observations are removed and no differencing is applied
automatically based on these results.

## Stationarity findings

At the nominal 5% level, ADF rejects the unit-root null for revenue,
Google Search, Radio and Outdoor. KPSS does not reject level
stationarity for these variables. The results are therefore
consistent with level stationarity under the chosen specifications.

For TV, Facebook/Instagram and YouTube, neither test rejects its
null hypothesis. These results are inconclusive and do not establish
either stationarity or non-stationarity.

All KPSS p-values exceed the upper tabulated bound of 0.10.

The sample contains only 30 observations. Results are exploratory,
sensitive to test specifications, and unadjusted for multiple testing.
They do not rule out seasonal variation.

No automatic differencing or removal of variables is performed.

In [ ]:
# Use the same first 30 months as the other exploratory analyses.
yoy = (
    eda[["month", config.TARGET]]
    .sort_values("month")
    .reset_index(drop=True)
    .copy()
)

# A 12-row shift represents 12 months only for consecutive monthly data.
expected_months = pd.date_range(
    start=yoy["month"].min(),
    periods=len(yoy),
    freq="MS",
)
assert pd.DatetimeIndex(yoy["month"]).equals(expected_months)

yoy["revenue_previous_year_lkr"] = (
    yoy[config.TARGET].shift(12)
)

yoy["revenue_yoy_growth_pct"] = (
    (
        yoy[config.TARGET]
        / yoy["revenue_previous_year_lkr"]
    )
    - 1
) * 100

yoy["comparison_available"] = (
    yoy["revenue_previous_year_lkr"].notna()
)

# The first 12 months have no previous-year comparison.
assert yoy["revenue_yoy_growth_pct"].iloc[:12].isna().all()
assert yoy["revenue_yoy_growth_pct"].iloc[12:].notna().all()
assert np.isfinite(
    yoy["revenue_yoy_growth_pct"].iloc[12:]
).all()

yoy_path = TABLE_DIR / "revenue_yoy_growth_train.csv"

yoy.to_csv(
    yoy_path,
    index=False,
    date_format="%Y-%m-%d",
    na_rep="",
)

available_yoy = yoy.loc[
    yoy["comparison_available"]
].copy()

fig, ax = plt.subplots(
    figsize=(12, 5),
    constrained_layout=True,
)

ax.plot(
    available_yoy["month"],
    available_yoy["revenue_yoy_growth_pct"],
    marker="o",
    color="#2563EB",
)

ax.axhline(
    0,
    color="black",
    linewidth=1,
    linestyle="--",
)

ax.set_title(
    "Year-on-year revenue growth | Jan 2024–Jun 2025\n"
    "Synthetic data; final six months excluded"
)
ax.set_xlabel("Month")
ax.set_ylabel("Revenue change versus same month last year (%)")

ax.xaxis.set_major_locator(
    mdates.MonthLocator(interval=2)
)
ax.xaxis.set_major_formatter(
    mdates.DateFormatter("%b %Y")
)

plt.setp(
    ax.get_xticklabels(),
    rotation=45,
    ha="right",
)

fig.savefig(
    FIGURE_DIR / "08_revenue_yoy_growth_train.png",
    dpi=200,
    bbox_inches="tight",
)

plt.show()
plt.close(fig)

display(
    available_yoy[
        [
            "month",
            config.TARGET,
            "revenue_previous_year_lkr",
            "revenue_yoy_growth_pct",
        ]
    ].round(2)
)

print("Saved:", yoy_path.relative_to(PROJECT_ROOT))
print("Total rows:", len(yoy))
print("Available YoY comparisons:", len(available_yoy))
print(
    "Expected missing YoY values:",
    yoy["revenue_yoy_growth_pct"].isna().sum(),
)

## Year-on-year revenue growth

The exploratory dataset provides 18 year-on-year comparisons,
covering January 2024 to June 2025. The first 12 months have no
previous-year observations, so their growth values remain missing.

Nine comparisons show positive growth and nine show negative growth.
The largest increase is January 2024 (+27.89%), while the largest
decline is August 2024 (-20.04%).

These changes describe total revenue movements, not incremental
advertising effects. Comparing the same calendar month does not
remove changes in promotions, prices, economic conditions or spend.

YoY growth is stored separately in
`outputs/tables/m1/revenue_yoy_growth_train.csv`.
The cleaned modelling dataset retains all 36 observations.

This revenue-derived measure must not be used as a contemporaneous
predictor of revenue.